In [ ]:
# Should be run in a virtual environment with both timesfm and dtaianomaly installed:
# pip install timesfm[torch]
# pip install dtaianomaly[notebook]

import timesfm
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from dtaianomaly.visualization import plot_demarcated_anomalies

In [ ]:
# The longer forecasting horizon used in Figure 6 of the paper, which will be compared to horizon = 1
prediction_length = 64

# Case corresponding to Figure 6(a) in the paper
file_name = '371_UCR_id_69_Medical_tr_1853_1st_6200.csv'
end = 6197 # specify the time step of the end of the context window

# Case corresponding to Figure 6(b) in the paper
# file_name = '558_YAHOO_id_8_WebService_tr_500_1st_1125.csv'
# end = 1130

context_length = 512

In [ ]:
# Read in data and labels

data_direc = f'../data/TSB-AD-U/{file_name}'
df = pd.read_csv(data_direc).dropna()
data = df.iloc[:, 0:-1].values.astype(float).squeeze()
label = df['Label'].astype(int).to_numpy()

# Initialize TimesFM with the longer forecasting horizon.

tfm = timesfm.TimesFm(
    hparams=timesfm.TimesFmHparams(
        backend="gpu" if torch.cuda.is_available() else "cpu",
        per_core_batch_size=32,
        horizon_len=prediction_length,
    ),
    checkpoint=timesfm.TimesFmCheckpoint(
        huggingface_repo_id=f"google/timesfm-1.0-200m-pytorch"),
)

# Forecast with TimesFM with the longer forecasting horizon.

seqs = torch.tensor(data[end-context_length:end], dtype=torch.float32).unsqueeze(dim=0)
predictions_long, _ = tfm.forecast(seqs)

# Initialize TimesFM with horizon_len = 1.

tfm = timesfm.TimesFm(
    hparams=timesfm.TimesFmHparams(
        backend="gpu" if torch.cuda.is_available() else "cpu",
        per_core_batch_size=32,
        horizon_len=1,
    ),
    checkpoint=timesfm.TimesFmCheckpoint(
        huggingface_repo_id=f"google/timesfm-1.0-200m-pytorch"),
)

# Forecast with TimesFM with horizon_len = 1.

predictions_short = np.zeros(prediction_length)

for i in range(prediction_length):
    seqs = torch.tensor(data[end-context_length+i:end+i], dtype=torch.float32).unsqueeze(dim=0)
    predictions, _ = tfm.forecast(seqs)
    predictions_short[i] = predictions.squeeze()

In [ ]:
# Compare the forecast to the actual data, and visualize with anomalies demarcated.

plt.rcParams['font.size'] = 18
fig, (ax1, ax2) = plt.subplots(2, figsize=(20, 6), sharex='all', height_ratios=[1, 1])

plot_demarcated_anomalies(data[end-context_length:end+prediction_length], label[end-context_length:end+prediction_length], ax=ax1)
ax1.plot(np.arange(context_length + prediction_length), data[end-context_length:end+prediction_length], color='tab:blue', label='Actual Data')
ax1.plot(np.arange(context_length, context_length+prediction_length), predictions_short.squeeze(), label='Forecast')
ax1.legend()

plot_demarcated_anomalies(data[end-context_length:end+prediction_length], label[end-context_length:end+prediction_length], ax=ax2)
ax2.plot(np.arange(context_length + prediction_length), data[end-context_length:end+prediction_length], color='tab:blue', label='Actual Data')
ax2.plot(np.arange(context_length, context_length+prediction_length), predictions_long.squeeze(), label='Forecast')
ax2.legend()

fontsize = 45
ax1.set_title(f'Forecasting Horizon = 1', fontsize=fontsize)
ax2.set_title(f'Forecasting Horizon = {prediction_length}', fontsize=fontsize)

fig.tight_layout()